In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from torch.autograd import Function
from torch.nn.parameter import Parameter
from torch import optim
from IPython import display
import time
import pickle 
import matplotlib.pyplot as plt
import random
import os

if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

In [ ]:
import numpy as np

def load_point_cloud_from_obj(filepath):
    vertices = []

    with open(filepath, 'r') as file:
        for line in file:
            if line.startswith('v '): 
                parts = line.strip().split()
                vertex = list(map(float, parts[1:4]))
                vertices.append(vertex)

    return np.array(vertices, dtype=np.float32)

# Load the provided .obj file
file_path = "elephant-reference.obj"
point_cloud1 = load_point_cloud_from_obj(file_path)
point_cloud1.shape
indices = np.random.choice(point_cloud1.shape[0], 2000, replace=False)
elepant = point_cloud1[indices]

point_cloud2 = load_point_cloud_from_obj('horse-01.obj')
point_cloud2.shape
indices = np.random.choice(point_cloud2.shape[0], 2000, replace=False)
horse = point_cloud2[indices]

point_cloud3 = load_point_cloud_from_obj('flam-reference.obj')
point_cloud3.shape
indices = np.random.choice(point_cloud3.shape[0], 2000, replace=False)
flam = point_cloud3[indices]

In [ ]:
figure = [elepant,horse,flam]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import sys
from SEINT.SEINT_numpy import SEINT
step_indices = [0, 20, 40, 60, 80, 100]
colors = ['#2C3E50', '#A2B9C8', '#36597D', '#D6E4F0',
'#E37C59', '#F5BCA9', '#C95D39', '#FFDAC6',
'#4A7C59', '#B4CFB0', '#6F9E84', '#D5E7D2']
fig = plt.figure(figsize=(24, 10))
gs_top = fig.add_gridspec(nrows=3, ncols=6, top=0.95, bottom=0.45, hspace=0.05, wspace=0.05)
gs_bottom = fig.add_gridspec(nrows=2, ncols=6, top=0.4, bottom=0.11, hspace=0.4, wspace=0.5)

for t in range(3):
        X = figure[t]

        loss3 = []
        loss4 = []
        name = ['Elepant','Horse','Flam']
        angle = [50,60,70]
        Z = [X]
        steps = np.linspace(0, 0.51, 101)  

        noise_std = 0.2  
        noise = np.random.normal(scale=noise_std, size=X.shape)  

        for idx, alpha in enumerate(steps):
                zi = (1-alpha)* X + alpha * noise  
                Z.append(zi)
                loss3.append(SEINT(zi,X,maxed=True, rep = 50, set_seed = 42,rd_rad = 3))
                loss4.append(SEINT(zi,X,maxed=False, rep = 50, set_seed = 42,rd_rad = 3))



        for j, idx0 in enumerate(step_indices):
                if (j==0):
                        ax = fig.add_subplot(gs_top[t, j], projection='3d')
                        zi = Z[idx0]  
                        ax.scatter(zi[:, 2], zi[:, 0], zi[:, 1], c=colors[4*t+1], s=0.5,alpha=0.8)
                        ax.set_title(f'Source shape : {name[t]}', fontsize=22, fontname='Times New Roman',fontweight='bold')
                        ax.view_init(elev=0, azim=angle[t])
                        ax.set_axis_off()
                else:
                        ax = fig.add_subplot(gs_top[t, j], projection='3d')
                        zi = Z[idx0]  
                        ax.scatter(zi[:, 2], zi[:, 0], zi[:, 1], c=colors[4*t], s=0.5,alpha=0.8)
                        ax.set_title(f'Step = {idx0}', fontsize=22, fontname='Times New Roman')
                        ax.view_init(elev=0, azim=angle[t])
                        ax.set_axis_off()

        ax = fig.add_subplot(gs_bottom[0:, 2*t:2*t+2])
        ax.plot(loss3, 
                color=colors[4*t+2], 
                linestyle='--', 
                linewidth=1, 
                marker='o', 
                markersize=3, 
                label='SEINT')

        ax.plot(loss4, 
                color=colors[4*t+3], 
                linestyle='-', 
                linewidth=1, 
                marker='x', 
                markersize=3, 
                label='ISEINT')
        ax.set_title(f'Loss Curve for {name[t]}',fontsize=22,fontname='Times New Roman')
        ax.set_xlabel('Step',fontsize=22,fontname='Times New Roman')
        ax.set_ylabel('Loss',fontsize=22,fontname='Times New Roman')
        ax.tick_params(axis='both', labelsize=20)
        ax.legend(fontsize=15, frameon=False, loc='lower right')
plt.tight_layout()
#fig.subplots_adjust(wspace=0.15, hspace=0.25)
plt.savefig("noise.png", dpi=300)
plt.show()